# KoBERT 를 이용한 영화리뷰 분류하기

- BERT(Bidirectional Encoder Representations from Transformers)는 Transformer의 인코더(Encoder) 부분만 사용하는 모델

- Transformer 원래 구조는 Encoder-Decoder로 이루어져 있는데, BERT는 그중 Encoder 스택만 쌓아서 만든 모델

- BERT의 목적은 문장을 잘 "이해"하는 표현(representation)을 학습하는 것이지, 새로운 텍스트를 순차적으로 "생성"하는 것이 아닙니다

- Encoder는 입력 전체를 양방향(bidirectional)으로 한 번에 보고 문맥을 파악할 수 있어서, 이해 중심 태스크(분류, NER, QA 등)에 적합

# 데이터 로드 및 정제

In [1]:
!pip install "transformers>=4.0,<5.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 126.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.28.0
    Uninstalling huggingface_hub-1.28.0:
      Successfully uninstalled huggingface_hub-1.28.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.23.1
    Uninstalling tokenizers-0.23.1:
      Successfully uninstalled tokenizers-0.23.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of t

In [2]:
import pandas as pd
import numpy as np
import urllib.request
import os
from tqdm import tqdm
import tensorflow as tf
from tensorflow import keras
from transformers import BertTokenizer, TFBertModel

In [3]:
# 네이버 영화 리뷰 데이터 학습을 위해 훈련 데이터와 테스트 데이터를 다운로드합니다.
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt",
                           filename="ratings_train.txt")
urllib.request.urlretrieve("https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt",
                           filename="ratings_test.txt")

('ratings_test.txt', <http.client.HTTPMessage at 0x7d2e6eeef950>)

In [4]:
train_data = pd.read_table('ratings_train.txt')
test_data = pd.read_table('ratings_test.txt')

print('훈련용 리뷰 개수:',len(train_data)) # 훈련 용 리 뷰 개 수 출 력
print('테스트용 리뷰 개수:',len(test_data)) # 테 스 트 용 리 뷰 개 수 출 력

훈련용 리뷰 개수: 150000
테스트용 리뷰 개수: 50000


In [5]:
train_data.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [6]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150000 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        150000 non-null  int64 
 1   document  149995 non-null  object
 2   label     150000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.4+ MB


In [7]:
test_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        50000 non-null  int64 
 1   document  49997 non-null  object
 2   label     50000 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.1+ MB


In [8]:
# 제거해야하는 중복된 데이터들 많음
train_data['document'].value_counts().head()

,count
document,
굿,181
good,92
최고,85
쓰레기,79
별로,66


In [9]:
train_data.loc[train_data['document'].isna()]  # 결측치 있다.  이 또한 제거

,id,document,label
25857,2172111,NaN,1
55737,6369843,NaN,1
110014,1034280,NaN,0
126782,5942978,NaN,0
140721,1034283,NaN,0


In [10]:
# 중복데이터와 결측치 제거
train_data.drop_duplicates(subset=['document'], inplace=True)  # document 컬럼에서 중복인 내용이 있다면 제거
train_data.dropna(how='any', inplace=True)  # NaN이 존재하는 행 제거

len(train_data)

146182

In [11]:
test_data.dropna(how='any', inplace=True)
len(test_data)

49997

# BERT 입력

In [12]:
# BERT 의 입력은 세가지를 준비해야 한다
#  1. 정수인코딩
#  2. 세그먼트 인코딩  (문장구분)
#  3. 어텐션 마스크 (단어토큰, 패딩토큰 구분)

In [13]:
tokenizer = BertTokenizer.from_pretrained('klue/bert-base')

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

In [14]:
sentence = "보는 내내 그대로 들어맞는 예측 카리스마 없는 악역"

In [15]:
tokenizer.tokenize(sentence)

['보', '##는', '내내', '그대로', '들어맞', '##는', '예측', '카리스마', '없', '##는', '악역']

In [16]:
encoded = tokenizer.encode(sentence)
encoded

[2, 1160, 2259, 6404, 4311, 20657, 2259, 5501, 13132, 1415, 2259, 23713, 3]

In [17]:
# 여기서 주의할 점은 앞의 2 번과 뒤에 붙은 3 번은 원래 있던 단어가 아니라는 점입니다.
# decode() 를 사용하면 정수 인코딩 결과를 다시 텍스트로 변환합니다.
# 2 -> [CLS]
# 3 -> [SEP]

In [18]:
tokenizer.decode(encoded)

'[CLS] 보는 내내 그대로 들어맞는 예측 카리스마 없는 악역 [SEP]'

In [19]:
2

2

In [20]:
print(tokenizer.cls_token, tokenizer.cls_token_id)
print(tokenizer.sep_token, tokenizer.sep_token_id)
print(tokenizer.pad_token, tokenizer.pad_token_id)

[CLS] 2
[SEP] 3
[PAD] 0


In [21]:
# encode() : 정수인코딩 + 패딩 동시에 가능
#   max_length= : 최대 인코딩 길이
#   padding='max_length' : 최대길이까지 패딩
#   truncation=True

## 정수 인코딩 + 패딩

In [22]:
max_seq_len = 128
encoded_result = tokenizer.encode(
    "전율을 일으키는 영화. 다시 보고싶은 영화",
    padding='max_length',
    max_length=max_seq_len,
    truncation=True,
)

print(encoded_result)
print('길이:', len(encoded_result))

[2, 1537, 2534, 2069, 6572, 2259, 3771, 18, 3690, 4530, 2585, 2073, 3771, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
길이: 128


## 세그먼트 인코딩

In [23]:
# 어차피 '하나의 텍스트'로 이루어져 있으니 0으로 채워서 준비한다

In [24]:
print([0] * max_seq_len)

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


## 어텐션 패딩 마스크 인코딩

In [25]:
# 인코딩된 토큰 -> 1
# 패딩 토큰 -> 0
valid_num = len(tokenizer.encode("전율을 일으키는 영화. 다시 보고싶은 영화"))
print(valid_num * [1] + (max_seq_len - valid_num) * [0])

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [26]:
# 입력된 전체 데이터에 대해 위 과정을 진행하는 함수 만들기

def convert_examples_to_features(examples, labels, max_seq_len, tokenizer):
  # input_ids: 워드 임베딩을 위한 문장의 정수인코딩
  # attention_masks: 어텐션 마스크 인코딩
  # token_type_ids: 세그먼트 인코딩
  input_ids, attention_masks, token_type_ids, data_labels = [], [], [], []

  for example, label in tqdm(zip(examples, labels), total=len(examples)):
    # input_id 는 워 드 임 베 딩 을 위 한 문 장 의 정 수 인 코 딩
    input_id = tokenizer.encode(example, padding='max_length', max_length=max_seq_len, truncation=True)
    # attention_mask 는 실 제 단 어 가 위 치 하 면 1, 패 딩 의 위 치 에 는 0 인 시 퀀 스 .
    padding_count = input_id.count(tokenizer.pad_token_id)
    attention_mask = [1] * (max_seq_len - padding_count) + [0] * padding_count
    # token_type_id 은 세 그 먼 트 인 코 딩
    token_type_id = [0] * max_seq_len

    assert len(input_id) == max_seq_len, "Error with input length {} vs {}".format(len(input_id), max_seq_len)
    assert len(attention_mask) == max_seq_len, "Error with attention mask length {} vs {}".format(len(attention_mask), max_seq_len)
    assert len(token_type_id) == max_seq_len, "Error with token type length {} vs {}".format(len(token_type_id), max_seq_len)

    input_ids.append(input_id)
    attention_masks.append(attention_mask)
    token_type_ids.append(token_type_id)
    data_labels.append(label)

  # TFBertModel 은 tf.Tensor 나 numpy 를 입력으로 기대함.
  # 간혹 list 를 전달해도 tf.Tensor 로 변환되기도 하나...
  # 많은 경우 실패함...
  input_ids = np.array(input_ids, dtype=int) # 정수인코딩 + 패딩
  attention_masks = np.array(attention_masks, dtype=int)  # 어텐션 마스크
  token_type_ids = np.array(token_type_ids, dtype=int)  # 세그먼트 인코딩

  data_labels = np.asarray(data_labels, dtype=np.int32)


  return (input_ids, attention_masks, token_type_ids), data_labels

In [27]:
# 훈련 데이터에 대해서 진행
train_X, train_y = convert_examples_to_features(train_data['document'], train_data['label'],
                                                max_seq_len=max_seq_len, tokenizer=tokenizer)

100%|██████████| 146182/146182 [00:37<00:00, 3911.88it/s]


In [28]:
# 테스트 데이터에 대해서 진행.
test_X, test_y = convert_examples_to_features(test_data['document'], test_data['label'],
                                              max_seq_len=max_seq_len, tokenizer=tokenizer)

100%|██████████| 49997/49997 [00:12<00:00, 4077.82it/s]


In [29]:
# 훈련데이터의 첫번째 샘플에 대해 출력

input_id = train_X[0][0]
attention_mask = train_X[1][0]
token_type_id = train_X[2][0]
label = train_y[0]

print('단어에 대한 정수 인코딩:', input_id)
print('어텐션 마스크:', attention_mask)
print('세그먼트 인코딩:', token_type_id)
print('각 인코딩 의 길이:', len(input_id))
print('정수 인코딩 복원:', tokenizer.decode(input_id))
print('레이블 :',label)

단어에 대한 정수 인코딩: [   2 1376  831 2604   18   18 4229 9801 2075 2203 2182 4243    3    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0]
어텐션 마스크: [1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
세그먼트 인코딩: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0

# BERT 출력

In [30]:
# 사전학습 모델
model = TFBertModel.from_pretrained('klue/bert-base', from_pt=True)

pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.bias', 'bert.embeddings.position_ids', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClas

In [31]:
# 파라미터 개수
model.count_params()

110617344

In [32]:
max_seq_len = 128    # 입력문장의 길이 128

input_ids_layer = tf.keras.layers.Input(shape=(max_seq_len,), dtype=tf.int32)
attention_masks_layer = tf.keras.layers.Input(shape=(max_seq_len,), dtype=tf.int32)
token_type_ids_layer = tf.keras.layers.Input(shape=(max_seq_len,), dtype=tf.int32)

# BERT의 출력 outputs
outputs = model([input_ids_layer, attention_masks_layer, token_type_ids_layer])

In [33]:
# outputs 에는 두 개의 출력이 존재하는데 각각 인덱스 0 과 1 로 접근하여 크기를 확인해봅시다.

print(outputs[0])

# (None, 128, 768)
#  문장의 길이 개수만큼 출력.  Many-To-Many 태스크인 경우 outputs[0] 을 사용.

print(outputs[1])
# (None, 768)
#     [CLS] 토큰 위치의 출력 Many-To-One 태스크인 경우 outputs[1]을 사용
#     지금과 같은 영화리뷰 분류 문제는 이에 해당

KerasTensor(type_spec=TensorSpec(shape=(None, 128, 768), dtype=tf.float32, name=None), name='tf_bert_model/bert/encoder/layer_._11/output/LayerNorm/batchnorm/add_1:0', description="created by layer 'tf_bert_model'")
KerasTensor(type_spec=TensorSpec(shape=(None, 768), dtype=tf.float32, name=None), name='tf_bert_model/bert/pooler/dense/Tanh:0', description="created by layer 'tf_bert_model'")


# BERT 를 이용한 Many-To-One 모델 만들기

In [34]:

class TFBertForSequenceClassification(tf.keras.Model):

  def __init__(self, model_name):
    super(TFBertForSequenceClassification, self).__init__()

    self.bert = TFBertModel.from_pretrained(model_name, from_pt=True)
    self.classifier = tf.keras.layers.Dense(
                                            1,  # 이진분류 문제
                                            # 가중치 초기화 (평균 0, 표준편차 0.02)
                                            kernel_initializer=tf.keras.initializers.TruncatedNormal(stddev=0.02),
                                            activation='sigmoid', name='classifier')

  def call(self, inputs):
    """순전파 정의"""
    input_ids, attention_mask, token_type_ids = inputs
    outputs = self.bert(input_ids=input_ids,
                        attention_mask=attention_mask,
                        token_type_ids=token_type_ids)

    cls_token = outputs[1]  # Many-To-One
    prediction = self.classifier(cls_token)

    return prediction

In [35]:
model = TFBertForSequenceClassification("klue/bert-base")
optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5)
loss = tf.keras.losses.BinaryCrossentropy()
model.compile(optimizer=optimizer, loss=loss, metrics = ['accuracy'])


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.bias', 'bert.embeddings.position_ids', 'cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the 

In [36]:
model.fit(train_X, train_y, epochs=2, batch_size=64, validation_split=0.2)

Epoch 1/2
1828/1828 [==============================] - 3128s 2s/step - loss: 0.2816 - accuracy: 0.8808 - val_loss: 0.2411 - val_accuracy: 0.9008
Epoch 2/2
1828/1828 [==============================] - 3093s 2s/step - loss: 0.1853 - accuracy: 0.9267 - val_loss: 0.2442 - val_accuracy: 0.9035


In [37]:
results = model.evaluate(test_X, test_y, batch_size=1024)
print("test loss, test acc: ", results)

49/49 [==============================] - 430s 9s/step - loss: 0.2536 - accuracy: 0.8984
test loss, test acc:  [0.2536034882068634, 0.8984339237213135]


# 예측

In [45]:
def sentiment_predict(new_sentence):

  # 1.정수인코딩 + 패딩
  input_id = tokenizer.encode(new_sentence,
                              padding='max_length', max_length=max_seq_len, truncation=True)

  # 2.어텐션 마스크
  padding_count = input_id.count(tokenizer.pad_token_id)
  attention_mask = [1] * (max_seq_len - padding_count) + [0] * padding_count

  # 3. 세그먼트 인코딩
  token_type_id = [0] * max_seq_len

  # 위 입력 데이터를 numpy 로 변환
  input_ids = np.array([input_id])  # <- 2차원 데이터로!
  attention_masks = np.array([attention_mask])
  token_type_ids = np.array([token_type_id])

  # 입력 시퀀스 준비
  encoded_input = [input_ids, attention_masks, token_type_ids]
  score = model.predict(encoded_input)[0][0]  # 첫번째 batch의 출력값 0.0 ~ 1.0

  if score > 0.5:
    print("{:.2f}% 확률로 😀긍정 리뷰입니다.\n".format(score * 100))
  else:
    print("{:.2f}% 확률로 👿부정 리뷰입니다.\n".format((1 - score) * 100))



In [46]:
input_sentiments = [
    '보던거라 계속보고있는데 전개도 느리고 주인공인 은희는 한두컷 나오면서 소극적인모습에 ',
    "스토리는 확실히 실망이였지만 배우들 연기력이 대박이였다 특히 이제훈 연기 정말 ... 이 배우들로 이렇게밖에 만들지 못한 영화는 아쉽지만 배우들 연기력과 사운드는 정말 빛났던 영화. 기대하고 극장에서 보면 많이 실망했겠지만 평점보고 기대없이 집에서 편하게 보면 괜찮아요. 이제훈님 연기력은 최고인 것 같습니다",
    "남친이 이 영화를 보고 헤어지자고한 영화. 자유롭게 살고 싶다고 한다. 내가 무슨 나비를 잡은 덫마냥 나에겐 다시 보고싶지 않은 영화.",
    "이 영화 존잼입니다 대박",
    '이 영화 개꿀잼 ㅋㅋㅋ',
    '이 영화 핵노잼 ㅠㅠ',
    '감독 뭐하는 놈이냐?',
    '와 개쩐다 정말 세계관 최강자들의 영화다',
]

In [47]:
for sentiment in input_sentiments:
  print(sentiment)
  sentiment_predict(sentiment)
  print('🟦' * 20)

보던거라 계속보고있는데 전개도 느리고 주인공인 은희는 한두컷 나오면서 소극적인모습에 
1/1 [==============================] - 0s 58ms/step
90.53% 확률로 👿부정 리뷰입니다.

🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
스토리는 확실히 실망이였지만 배우들 연기력이 대박이였다 특히 이제훈 연기 정말 ... 이 배우들로 이렇게밖에 만들지 못한 영화는 아쉽지만 배우들 연기력과 사운드는 정말 빛났던 영화. 기대하고 극장에서 보면 많이 실망했겠지만 평점보고 기대없이 집에서 편하게 보면 괜찮아요. 이제훈님 연기력은 최고인 것 같습니다
1/1 [==============================] - 0s 57ms/step
94.37% 확률로 😀긍정 리뷰입니다.

🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
남친이 이 영화를 보고 헤어지자고한 영화. 자유롭게 살고 싶다고 한다. 내가 무슨 나비를 잡은 덫마냥 나에겐 다시 보고싶지 않은 영화.
1/1 [==============================] - 0s 55ms/step
97.36% 확률로 👿부정 리뷰입니다.

🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
이 영화 존잼입니다 대박
1/1 [==============================] - 0s 53ms/step
97.98% 확률로 😀긍정 리뷰입니다.

🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
이 영화 개꿀잼 ㅋㅋㅋ
1/1 [==============================] - 0s 58ms/step
98.01% 확률로 😀긍정 리뷰입니다.

🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
이 영화 핵노잼 ㅠㅠ
1/1 [==============================] - 0s 57ms/step
99.38% 확률로 👿부정 리뷰입니다.

🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦🟦
감독 뭐하는 놈이냐?
1/1 [==============================] - 0s 57ms/step
99.14% 확률로 👿부정 리뷰입니다.

🟦